In [1]:
"""
Bai06 - Lab thực hành Chương 15: Đưa GenAI vào Hoạt động: Ghi nhật ký, Giám sát và Lỗi
Sách: Supercharged Coding with Gen AI

Mục tiêu:
  - Thêm logging, monitoring, error handling vào FizzBuzz
  - Dùng decorator để tách biệt concerns (Single Responsibility Principle)
  - Minh họa Chain-of-Thought (CoT) prompt với few-shot learning

Prompt mẫu (Chain-of-Thought):
  Chain 1: "Implement FizzBuzz as a clean function"
  Chain 2: "Add a logging decorator that logs function name and args"
  Chain 3: "Add a call counter decorator"
  Chain 4: "Add input validation decorator for positive int < 500"
  (Tách thành 4 prompt riêng biệt → clean code, không vi phạm SRP)
"""

import logging
import functools
import time
from collections import defaultdict

# ─────────────────────────────────────────────
# Logging setup
# ─────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(name)s - %(message)s",
    handlers=[
        logging.StreamHandler(),  # Console
        # logging.FileHandler("fizzbuzz.log")  # uncommment để ghi file
    ]
)
logger = logging.getLogger(__name__)

# ─────────────────────────────────────────────
# Global call counter (for monitoring)
# ─────────────────────────────────────────────
_call_counts: dict = defaultdict(int)


# ─────────────────────────────────────────────
# Decorator 1: Logging (không làm lẫn với logic chính)
# GenAI Prompt: "Write a decorator that logs function name and arguments"
# ─────────────────────────────────────────────
def log_call(func):
    """Decorator: log function name and arguments on each call.

    Applies the Single Responsibility Principle – logging is separated
    from the function's core logic.

    Args:
        func: The function to wrap.

    Returns:
        Callable: Wrapped function with logging.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        logger.info(f"Calling {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        logger.debug(f"{func.__name__} completed.")
        return result
    return wrapper


# ─────────────────────────────────────────────
# Decorator 2: Call counter (monitoring)
# GenAI Prompt: "Write a decorator that counts and reports total calls"
# ─────────────────────────────────────────────
def count_calls(func):
    """Decorator: count total calls to the function.

    Args:
        func: The function to wrap.

    Returns:
        Callable: Wrapped function with call counting.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        _call_counts[func.__name__] += 1
        return func(*args, **kwargs)
    return wrapper


# ─────────────────────────────────────────────
# Decorator 3: Input validation
# GenAI Prompt (CoT): "Write a decorator that validates limit is a
# positive integer less than 500, raises TypeError/ValueError otherwise"
# ─────────────────────────────────────────────
def validate_fizzbuzz_input(func):
    """Decorator: validate that limit is a positive integer < 500.

    Raises:
        TypeError: If limit is not an integer.
        ValueError: If limit <= 0 or limit >= 500.

    Args:
        func: The function to wrap.

    Returns:
        Callable: Wrapped function with input validation.
    """
    @functools.wraps(func)
    def wrapper(limit, *args, **kwargs):
        if not isinstance(limit, int):
            raise TypeError(f"limit must be int, got {type(limit).__name__}")
        if limit <= 0 or limit >= 500:
            raise ValueError(f"limit must be in range (0, 500), got {limit}")
        return func(limit, *args, **kwargs)
    return wrapper


# ─────────────────────────────────────────────
# Decorator 4: Performance timer
# ─────────────────────────────────────────────
def timeit(func):
    """Decorator: measure and log execution time.

    Args:
        func: The function to wrap.

    Returns:
        Callable: Wrapped function with timing.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed_ms = (time.perf_counter() - start) * 1000
        logger.debug(f"{func.__name__} took {elapsed_ms:.3f}ms")
        return result
    return wrapper


# ─────────────────────────────────────────────
# BEFORE: FizzBuzz với logging lẫn lộn (BAD)
# Vi phạm Single Responsibility Principle
# ─────────────────────────────────────────────
def print_fizzbuzz_cluttered(limit: int) -> None:
    """BAD example: logging mixed into core logic (violates SRP)."""
    # Clutter: validation + logging inline
    if not isinstance(limit, int):
        raise TypeError("limit must be int")
    logger.info(f"print_fizzbuzz called with limit={limit}")
    _call_counts['print_fizzbuzz_cluttered'] += 1

    for i in range(1, limit + 1):
        if i % 15 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)


# ─────────────────────────────────────────────
# AFTER: FizzBuzz sạch + decorators tách biệt (GOOD)
# GenAI Chain-of-Thought approach
# ─────────────────────────────────────────────
@log_call
@count_calls
@validate_fizzbuzz_input
@timeit
def print_fizzbuzz(limit: int) -> list[str]:
    """Generate FizzBuzz sequence up to limit.

    Core logic is clean and focused. All cross-cutting concerns
    (logging, counting, validation, timing) handled by decorators.

    Args:
        limit (int): Upper bound of the FizzBuzz sequence (1 to limit).
            Must be a positive integer less than 500.

    Returns:
        list[str]: FizzBuzz sequence as a list of strings.
    """
    result = []
    for i in range(1, limit + 1):
        if i % 15 == 0:
            result.append("FizzBuzz")
        elif i % 3 == 0:
            result.append("Fizz")
        elif i % 5 == 0:
            result.append("Buzz")
        else:
            result.append(str(i))
    return result


def get_call_stats() -> dict:
    """Return call statistics for all monitored functions.

    Returns:
        dict: Mapping of function name to call count.
    """
    return dict(_call_counts)


# ─────────────────────────────────────────────
# Demo chạy
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("Ch15 Lab: Going Live with GenAI - Logging & Monitoring")
    print("=" * 55)

    print("\n[1] FizzBuzz with decorators (limit=20):")
    seq = print_fizzbuzz(20)
    print(f"  Result: {seq}")

    print("\n[2] Calling multiple times for monitoring:")
    print_fizzbuzz(10)
    print_fizzbuzz(15)

    print(f"\n[3] Call statistics: {get_call_stats()}")

    print("\n[4] Input validation tests:")
    try:
        print_fizzbuzz("twenty")
    except TypeError as e:
        print(f"  TypeError caught: {e}")

    try:
        print_fizzbuzz(0)
    except ValueError as e:
        print(f"  ValueError caught: {e}")

    try:
        print_fizzbuzz(600)
    except ValueError as e:
        print(f"  ValueError caught: {e}")

    print("\n[5] Decorator chain order matters:")
    print("  log_call → count_calls → validate → timeit → core function")
    print("  Each decorator adds ONE cross-cutting concern (SRP compliant)")


2026-03-25 17:00:41,805 [INFO] __main__ - Calling print_fizzbuzz((20,), {})
2026-03-25 17:00:41,806 [DEBUG] __main__ - print_fizzbuzz took 0.005ms
2026-03-25 17:00:41,807 [DEBUG] __main__ - print_fizzbuzz completed.
2026-03-25 17:00:41,808 [INFO] __main__ - Calling print_fizzbuzz((10,), {})
2026-03-25 17:00:41,808 [DEBUG] __main__ - print_fizzbuzz took 0.004ms
2026-03-25 17:00:41,808 [DEBUG] __main__ - print_fizzbuzz completed.
2026-03-25 17:00:41,809 [INFO] __main__ - Calling print_fizzbuzz((15,), {})
2026-03-25 17:00:41,809 [DEBUG] __main__ - print_fizzbuzz took 0.005ms
2026-03-25 17:00:41,810 [DEBUG] __main__ - print_fizzbuzz completed.
2026-03-25 17:00:41,810 [INFO] __main__ - Calling print_fizzbuzz(('twenty',), {})
2026-03-25 17:00:41,810 [INFO] __main__ - Calling print_fizzbuzz((0,), {})
2026-03-25 17:00:41,811 [INFO] __main__ - Calling print_fizzbuzz((600,), {})


Ch15 Lab: Going Live with GenAI - Logging & Monitoring

[1] FizzBuzz with decorators (limit=20):
  Result: ['1', '2', 'Fizz', '4', 'Buzz', 'Fizz', '7', '8', 'Fizz', 'Buzz', '11', 'Fizz', '13', '14', 'FizzBuzz', '16', '17', 'Fizz', '19', 'Buzz']

[2] Calling multiple times for monitoring:

[3] Call statistics: {'print_fizzbuzz': 3}

[4] Input validation tests:
  TypeError caught: limit must be int, got str
  ValueError caught: limit must be in range (0, 500), got 0
  ValueError caught: limit must be in range (0, 500), got 600

[5] Decorator chain order matters:
  log_call → count_calls → validate → timeit → core function
  Each decorator adds ONE cross-cutting concern (SRP compliant)
